In [1]:
import pandas as pd
import json
import re
from pathlib import Path

## 1. Reading Datasets

In [2]:
BASE_DIR = Path.cwd().parent 
data_dir = BASE_DIR / "data"/"raw"

train_df = pd.read_parquet(data_dir / "train-00000-of-00001.parquet")
dev_df   = pd.read_parquet(data_dir / "dev-00000-of-00001.parquet")
test_df  = pd.read_parquet(data_dir / "test-00000-of-00001.parquet")

## 2. Text Normalization and Cleaning

In [3]:
def normalize_text(text):
    if not isinstance(text, str):
        return text
    
    text = re.sub(r'\s+', ' ', text).strip() #remove extra whitespace 
    
    text = re.sub(r'http\S+|www\S+', '', text)#  remove URLs 
    
    text = re.sub(r'\S+@\S+', '', text) # remove email addresses 
    
    text = re.sub(r'[^\w\s\.\,\!\?\'\"]', '', text) # remove special characters
    
    return text.strip()

## 3. Schema Validation

In [4]:
def validate_output_schema(output_json):
    try:
        if "aspects" not in output_json:
            return False, "Missing required field: 'aspects'"
        
        if not isinstance(output_json["aspects"], list):
            return False, "Field 'aspects' must be a list"
        
        for i, aspect in enumerate(output_json["aspects"]):
            if not isinstance(aspect, dict):
                return False, f"Aspect {i} is not a dictionary"

            if "term" not in aspect or "polarity" not in aspect:
                return False, f"Aspect {i} missing required fields. Expected: 'term' and 'polarity'"

            if not isinstance(aspect["term"], str):
                return False, f"Aspect {i}: 'term' must be a string"
            if not isinstance(aspect["polarity"], str):
                return False, f"Aspect {i}: 'polarity' must be a string"
        
        return True, "Valid"
    
    except Exception as e:
        return False, str(e)

test_valid_json = {
    "aspects": [
        {"term": "cord", "polarity": "neutral"},
        {"term": "battery life", "polarity": "positive"}
    ]
}

test_invalid_json = {
    "aspects": [
        {"term": "quality"}
    ]
}

test_invalid_json_2 = {
    "missing_aspects": "field"
}

print("Valid schema:", validate_output_schema(test_valid_json))
print("Invalid schema (missing polarity):", validate_output_schema(test_invalid_json))
print("Invalid schema (missing aspects):", validate_output_schema(test_invalid_json_2))


Valid schema: (True, 'Valid')
Invalid schema (missing polarity): (False, "Aspect 0 missing required fields. Expected: 'term' and 'polarity'")
Invalid schema (missing aspects): (False, "Missing required field: 'aspects'")


## 4. Data Cleaning and Filtering

In [5]:
def clean_dataset(df, dataset_name="dataset"):
    report = {
        "name": dataset_name,
        "original_rows": len(df),
        "duplicates_removed": 0,
        "invalid_json_removed": 0,
        "invalid_schema_removed": 0,
        "final_rows": 0
    }
    
    df_cleaned = df.drop_duplicates().reset_index(drop=True)
    report["duplicates_removed"] = len(df) - len(df_cleaned)
    
    valid_rows = []
    invalid_rows_list = []
    
    for idx, row in df_cleaned.iterrows():
        try:
            output_json = json.loads(row["output"])
            is_valid, msg = validate_output_schema(output_json)
            
            if is_valid:
                if "input" in row and isinstance(row["input"], str):
                    row["input"] = normalize_text(row["input"])
                valid_rows.append(row)
            else:
                invalid_rows_list.append((idx, "Invalid schema", msg))
                report["invalid_schema_removed"] += 1
        except json.JSONDecodeError as e:
            invalid_rows_list.append((idx, "Invalid JSON", str(e)))
            report["invalid_json_removed"] += 1
    
    df_cleaned = pd.DataFrame(valid_rows).reset_index(drop=True)
    report["final_rows"] = len(df_cleaned)
    report["invalid_rows_detail"] = invalid_rows_list[:10]
    
    return df_cleaned, report

print("=" * 60)
print("CLEANING DATASETS")
print("=" * 60 + "\n")

train_cleaned, train_report = clean_dataset(train_df, "Train")
dev_cleaned, dev_report = clean_dataset(dev_df, "Dev")
test_cleaned, test_report = clean_dataset(test_df, "Test")

for report in [train_report, dev_report, test_report]:
    print(f"\n{report['name']} Dataset Report:")
    print(f"  Original rows: {report['original_rows']}")
    print(f"  Duplicates removed: {report['duplicates_removed']}")
    print(f"  Invalid JSON removed: {report['invalid_json_removed']}")
    print(f"  Invalid schema removed: {report['invalid_schema_removed']}")
    print(f"  Final rows: {report['final_rows']}")
    if report['original_rows'] > 0:
        print(f"  Retention rate: {report['final_rows'] / report['original_rows'] * 100:.2f}%")


CLEANING DATASETS


Train Dataset Report:
  Original rows: 5959
  Duplicates removed: 0
  Invalid JSON removed: 0
  Invalid schema removed: 0
  Final rows: 5959
  Retention rate: 100.00%

Dev Dataset Report:
  Original rows: 200
  Duplicates removed: 0
  Invalid JSON removed: 0
  Invalid schema removed: 0
  Final rows: 200
  Retention rate: 100.00%

Test Dataset Report:
  Original rows: 1572
  Duplicates removed: 0
  Invalid JSON removed: 0
  Invalid schema removed: 0
  Final rows: 1572
  Retention rate: 100.00%


## 5. Save Processed Data

In [6]:
BASE_DIR = Path.cwd().parent 
processed_dir = BASE_DIR / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print("Saving cleaned datasets...")
train_cleaned.to_parquet(processed_dir / "train_cleaned.parquet")
dev_cleaned.to_parquet(processed_dir / "dev_cleaned.parquet")
test_cleaned.to_parquet(processed_dir / "test_cleaned.parquet")
print("Cleaned data saved as parquet files")



Saving cleaned datasets...
Cleaned data saved as parquet files


## 6. Sample Verification - Visual Inspection

In [7]:
print("=" * 70)
print("SAMPLE CLEANED DATA - TRAIN SET")
print("=" * 70 + "\n")

for sample_idx in range(min(3, len(train_cleaned))):
    print(f"\n--- Sample {sample_idx + 1} ---")
    row = train_cleaned.iloc[sample_idx]
    output_json = json.loads(row["output"])
    print(f"Input (Review): {row['input'][:100]}...")
    print(f"Domain: {row.get('domain', 'N/A')}")
    print(f"Aspects found: {len(output_json.get('aspects', []))}")
    for aspect in output_json.get('aspects', [])[:2]:
        print(f"  - Term: {aspect.get('term')} | Polarity: {aspect.get('polarity')}")


SAMPLE CLEANED DATA - TRAIN SET


--- Sample 1 ---
Input (Review): I charge it at night and skip taking the cord with me because of the good battery life....
Domain: laptops
Aspects found: 2
  - Term: cord | Polarity: neutral
  - Term: battery life | Polarity: positive

--- Sample 2 ---
Input (Review): I bought a HP Pavilion DV41222nr laptop and have had so many problems with the computer....
Domain: laptops
Aspects found: 0

--- Sample 3 ---
Input (Review): The tech guy then said the service center does not do 1to1 exchange and I have to direct my concern ...
Domain: laptops
Aspects found: 3
  - Term: service center | Polarity: negative
  - Term: "sales" team | Polarity: negative
